In [1]:
import os

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sqlite3 import connect
from pandas import DataFrame

from Constants import Constants as const

In [2]:
tqdm.pandas()

# Load latest NLRB data

In [9]:
nlrb_con = connect(os.path.join(const.DATABASE_PATH, 'union', 'NLRB', 'nlrb.db'))
# 1. 从 election 表中读取所有所需变量
election_raw = pd.read_sql("""
    SELECT election_id, case_number, date, tally_type
    FROM election
""", nlrb_con, parse_dates=['date'])

# 根据 case_number 和 tally_type 选择“最终有效结果”
def select_final_election(df):
    # 优先选择 run-off，其次 rerun，其次 initial
    for t_type in ['runoff', 'rerun', 'initial']:
        sub = df[df['tally_type'].str.lower() == t_type]
        if not sub.empty:
            return sub.sort_values('date').iloc[-1]  # 选择最后一次
    return df.sort_values('date').iloc[-1]  # 万一都没匹配上，退而求其次

election_df = election_raw.groupby('case_number', group_keys=False).apply(select_final_election).reset_index(drop=True)


# 2. 获取 filing 表数据
filing_df = pd.read_sql("""
    SELECT case_number, name, case_type, city, state
    FROM filing
""", nlrb_con)


# 3. 获取 tally 表数据
tally_df = pd.read_sql("""
    SELECT election_id, option, votes
    FROM tally
""", nlrb_con)

# 替换 votes 中的缺失值为 0
tally_df['votes'] = tally_df['votes'].fillna(0)

C:\Users\wangy\AppData\Local\Temp\ipykernel_24964\2532917584.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  election_df = election_raw.groupby('case_number', group_keys=False).apply(select_final_election).reset_index(drop=True)


In [4]:
# 每个 election_id 中票数最多的作为 vote_for，其余的作为 vote_against
def summarize_votes(group):
    votes_by_option = group.groupby('option')['votes'].sum()
    vote_for = votes_by_option.drop(labels='No union', errors='ignore').max() if not votes_by_option.drop(labels='No union', errors='ignore').empty else 0
    vote_against = votes_by_option.drop(labels='No union', errors='ignore').sum() - vote_for
    vote_no_union = votes_by_option.get('No union', 0)
    vote_against += vote_no_union
    return pd.Series({'vote_for': vote_for, 'vote_against': vote_against})

vote_summary_df = tally_df.groupby('election_id').progress_apply(summarize_votes).reset_index()


  0%|          | 0/32291 [00:00<?, ?it/s]

In [11]:
# 4. 合并所有数据
merged_df = election_df.merge(filing_df, on='case_number', how='left') \
                       .merge(vote_summary_df, on='election_id', how='left')

# 重命名变量
merged_df = merged_df.rename(columns={
    'case_number': 'casenumber',
    'name': 'employer',
    'case_type': 'type',
    'date': 'election_date',
})

merged_df['number_of_votes'] = merged_df['vote_for'] + merged_df['vote_against']

In [16]:
merged_df['election_date'].describe()

count                            30504
mean     2015-12-02 06:13:41.400471808
min                1994-03-04 00:00:00
25%                2011-01-07 00:00:00
50%                2015-07-22 12:00:00
75%                2020-12-14 00:00:00
max                2025-06-18 00:00:00
Name: election_date, dtype: object

In [17]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))

In [3]:
merged_df = pd.read_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))

## add address information

In [15]:
from rapidfuzz import process, fuzz  # 更快且推荐使用 rapidfuzz
import warnings
warnings.filterwarnings('ignore')

conn = connect(os.path.join(const.DATABASE_PATH, 'union', 'NLRB', 'nlrb.db'))

# 7. 获取 participant 表中的信息
participant_df = pd.read_sql("""
    SELECT case_number, participant, type, subtype, city,
           address_1, address_2, state, zip
    FROM participant
""", conn)

conn.close()

# 保留 type/subtype 中与 employer 相关的行
employer_participants = participant_df[
    participant_df['subtype'].str.lower().str.contains('employer', na=False)
].copy()

# 建立雇主地址参考表
employer_address_map = {}
for idx, row in tqdm(merged_df.iterrows()):
    case = row['casenumber']
    emp_name = row['employer']

    # 在相同 case_number 内匹配最相近的 participant
    subset = employer_participants[(employer_participants['case_number'] == case) & employer_participants['participant'].notnull()]
    if subset.shape[0] == 1:
        match_row = subset.iloc[0]
        employer_address_map[case] = {
            'employercity': match_row.get('city'),
            'employeraddress1': match_row.get('address_1'),
            'employeraddress2': match_row.get('address_2'),
            'employerstate': match_row.get('state'),
            'employerzip': match_row.get('zip'),
        }
    elif not subset.empty:
        # fuzzy match
        match, score, match_idx = process.extractOne(emp_name, subset['participant'], scorer=fuzz.token_sort_ratio)

        if score >= 80:  # 匹配阈值可调
            match_row = subset.loc[match_idx]
            employer_address_map[case] = {
                'employercity': match_row.get('city'),
                'employeraddress1': match_row.get('address_1'),
                'employeraddress2': match_row.get('address_2'),
                'employerstate': match_row.get('state'),
                'employerzip': match_row.get('zip'),
            }

# 转换为 DataFrame 并合并
employer_address_df = pd.DataFrame.from_dict(employer_address_map, orient='index').reset_index()
employer_address_df = employer_address_df.rename(columns={'index': 'casenumber'})

# 合并到 merged_df
merged_df = merged_df.merge(employer_address_df, on='casenumber', how='left')

# 示例输出
print(merged_df[['casenumber', 'employer', 'employeraddress1', 'employerstate']].head())


0it [00:00, ?it/s]

     casenumber                               employer     employeraddress1  \
0  01-RC-020966  New England Mechanical Services of MA        Washington St   
1  01-RC-020986          Courtyard Nursing Care Center    200 Governors Ave   
2  01-RC-021152                        Carney Hospital  2100 Dorchester Ave   
3  01-RC-021561                        Telcom USA Inc.       309 Andover St   
4  01-RC-022002  Regional Transportation Program, Inc.    127 Saint John St   

  employerstate  
0            MA  
1            MA  
2            MA  
3            MA  
4            ME  


In [17]:
merged_df.head()

,election_id,casenumber,election_date,tally_type,employer,type,city,state,vote_for,vote_against,number_of_votes,employercity,employeraddress1,employeraddress2,employerstate,employerzip
0,1,01-RC-020966,1999-03-17,Initial,New England Mechanical Services of MA,RC,Woburn,MA,0.0,0.0,0.0,Boston,Washington St,None,MA,-
1,2,01-RC-020986,1999-04-22,Initial,Courtyard Nursing Care Center,RC,Medford,MA,101.0,47.0,148.0,Medford,200 Governors Ave,None,MA,02155-1644
2,3,01-RC-021152,2000-04-27,Initial,Carney Hospital,RC,Boston,MA,0.0,229.0,229.0,Dorchester,2100 Dorchester Ave,None,MA,02124-5615
3,4,01-RC-021561,2002-11-07,Initial,Telcom USA Inc.,RC,Lawrence,MA,47.0,52.0,99.0,Lawrence,309 Andover St,None,MA,01843-2227
4,5,01-RC-022002,2007-04-25,Initial,"Regional Transportation Program, Inc.",RC,Portland,ME,5.0,4.0,9.0,Portland,127 Saint John St,None,ME,04102-3042


In [19]:
merged_df.loc[:, 'employercity'] = merged_df['employercity'].fillna(merged_df['city'])
merged_df.loc[:, 'employerstate'] = merged_df['employerstate'].fillna(merged_df['state'])


In [20]:
merged_df.to_pickle(os.path.join(const.TEMP_PATH, '1994_2025_union_election_information.pkl'))
